### Geometry

A circle of radius 1 with two circles of radius 1/8 removed from it.

In [ ]:
import gmsh
import numpy as np
from numpy import pi as π

gmsh.initialize()
geo = gmsh.model.geo

lcar = 1/32
radius = 1.0
deltas = np.array([(+1, 0), (0, +1), (-1, 0), (0, -1)])
origin = geo.add_point(0, 0, 0, lcar)
points = [geo.add_point(*(radius * delta), 0, lcar) for delta in deltas]
arcs = [
    geo.add_circle_arc(p1, origin, p2)
    for p1, p2 in zip(points, np.roll(points, 1))
]
geo.add_physical_group(1, arcs)
outer_curve_loop = geo.add_curve_loop(arcs)

centers = np.array([(0, +1/2), (0, -1/2)])
radii = [1/8, 1/8]
hole_curve_loops = []
for center, radius in zip(centers, radii):
    hole_center = geo.add_point(*center, 0, lcar)
    hole_points = [
        geo.add_point(*(center + radius * delta), 0, lcar) for delta in deltas
    ]
    hole_arcs = [
        geo.add_circle_arc(p1, hole_center, p2)
        for p1, p2 in zip(hole_points, np.roll(hole_points, 1))
    ]
    geo.add_physical_group(1, hole_arcs)
    curve_loop = geo.add_curve_loop(hole_arcs)
    hole_curve_loops.append(curve_loop)

plane_surface = geo.add_plane_surface([outer_curve_loop] + hole_curve_loops)
geo.add_physical_group(2, [plane_surface])
geo.synchronize()

gmsh.model.mesh.generate(2)
gmsh.write("mixer.msh")

gmsh.finalize()

In [ ]:
import firedrake
mesh = firedrake.Mesh('mixer.msh')

We'll set the velocity on the outer boundary to be 0, while the inner two circles are rotating with a fixed speed of 1.

In [ ]:
from firedrake import Constant, as_vector
x = firedrake.SpatialCoordinate(mesh)

x2 = Constant((0, +1/2))
r2 = Constant(1/8)
x3 = Constant((0, -1/2))
r3 = Constant(1/8)

q2 = (x - x2) / r2
q3 = (x - x3) / r3

u_Γ = (
    as_vector((0., 0.)),
    as_vector((-q2[1], q2[0])),
    as_vector((-q3[1], q3[0]))
)

### Solution

Now we can define our problem in much the same way as the last demo.
We'll use the same function spaces, viscosity, and solver parameters.

In [ ]:
Q = firedrake.FunctionSpace(mesh, family='CG', degree=1)
V = firedrake.VectorFunctionSpace(mesh, family='CG', degree=2)
Z = V * Q

And here we'll set the physical parameters for the problem.
For the Stokes equations to be reasonable, we need a much larger viscosity than the product of the domain size and the characteristic speed, so we're using $\mu = 10^3$.
We've also chosen a value of the friction coefficient $\kappa$ so that the ratio of the frictional length scale to the domain size is roughly equal to 2.

In [ ]:
from firedrake import (
    inner, grad, dx, ds, sym, div, derivative,
    MixedVectorSpaceBasis, VectorSpaceBasis, DirichletBC
)

def ε(u):
    return sym(grad(u))

n = firedrake.FacetNormal(mesh)

μ = Constant(1e3)
L = Constant(2.)

The objective functional has more parts than before, so to make the algebra more tractable we'll make separate variables for each summand.

In [ ]:
z = firedrake.Function(Z)
u, p = firedrake.split(z)

F = (μ * inner(ε(u), ε(u)) - p * div(u)) * dx

The solver parameters specify a direct factorization with MUMPS, which works well for 2D problems but less so for large 3D ones.

In [ ]:
bcs = [DirichletBC(Z.sub(0), u_γ, index) for index, u_γ in zip([1, 2, 3], u_Γ)]
basis = VectorSpaceBasis(constant=True, comm=firedrake.COMM_WORLD)
nullspace = MixedVectorSpaceBasis(Z, [Z.sub(0), basis])

firedrake.solve(
    derivative(F, z) == 0, z,
    nullspace=nullspace,
    bcs=bcs,
)

u, p = z.subfunctions

Finally we can plot the results.
Compared to the outcome in the last demo where we fixed the velocity around the boundaries, the counter-rotating vortices to either side of the hyperbolic fixed point at the origin have vanished.
If we increase the friction coefficient by a factor of 10 they reappear.
It would be a fun exercise in bifurcation theory to see at what exact value of $\kappa$ the vortices appear.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
fig, axes = plt.subplots()
axes.set_axis_off()
axes.set_aspect('equal')
kwargs = {'resolution': 1/30, 'seed': 0, 'cmap': 'winter'}
firedrake.streamplot(u, axes=axes, **kwargs);

As another experiment, you can re-run this notebook but reverse the direction of one of the mixer heads, which will remove the fixed point at the origin but will create two more on either side of it.